# 🐍 Master Guide: Structural Design Patterns in Python
> **Single-Source Revision Guide**: Covers Adapter, Decorator, Facade, Composite, and Proxy patterns.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Structural patterns ease design by identifying a simple way to realize relationships among entities: adapting incompatible interfaces, wrapping extra behaviors, or simplifying complex subsystems.

---
## 🟢 2. Part 1: Beginner Fundamentals

### Step 1: The Adapter Pattern
Translates an existing class interface into another interface callers expect.

In [ ]:
# Legacy third-party library interface:
class LegacyPaymentGateway:
    def make_payment(self, amount_cents):
        return f"Processed ${amount_cents / 100:.2f} via Legacy System"

# Modern client expects .pay(dollars):
class PaymentAdapter:
    def __init__(self, legacy_service):
        self.legacy = legacy_service
    
    def pay(self, dollars: float):
        cents = int(dollars * 100)
        return self.legacy.make_payment(cents)

adapter = PaymentAdapter(LegacyPaymentGateway())
print(adapter.pay(49.99))

---
## 🟡 3. Part 2: Intermediate Features

### The Facade Pattern
Provides a simplified, high-level interface over a complex ecosystem of subsystems.

In [ ]:
# Subsystems:
class CPU:
    def freeze(self): return "Freezing CPU"
    def jump(self, position): return f"Jumping to {position}"
    def execute(self): return "Executing instructions"

class Memory:
    def load(self, position, data): return f"Loading {data} into {position}"

class HardDrive:
    def read(self, lba, size): return f"Reading {size} bytes from sector {lba}"

# Unified Facade:
class ComputerFacade:
    def __init__(self):
        self.cpu = CPU()
        self.memory = Memory()
        self.hd = HardDrive()
    
    def start(self):
        print(self.cpu.freeze())
        print(self.memory.load("0x00", self.hd.read(100, 1024)))
        print(self.cpu.jump("0x00"))
        print(self.cpu.execute())
        print("Computer boot complete!")

comp = ComputerFacade()
comp.start()

### The Proxy Pattern (Lazy Loading & Access Control)

In [ ]:
# Proxy Pattern with lazy loading:
class RealVideoFile:
    def __init__(self, filename):
        self.filename = filename
        print(f"[Disk I/O] Loading 2GB video file '{filename}' into RAM...")
    
    def play(self):
        return f"Playing video: {self.filename}"

class VideoProxy:
    def __init__(self, filename):
        self.filename = filename
        self._real_video = None  # Loaded lazily only on first play
    
    def play(self):
        if self._real_video is None:
            self._real_video = RealVideoFile(self.filename)
        return self._real_video.play()

proxy = VideoProxy("movie_4k.mp4")
print("Video proxy created without disk overhead!")
print(proxy.play())  # Loads and plays on demand

---
## 📋 4. Part 3: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Pattern                            | Intent                                |
+------------------------------------+---------------------------------------+
| Adapter                            | Converts incompatible interface       |
| Facade                             | Simplified single entrypoint to system|
| Proxy                              | Lazy loading, access control placeholder|
| Composite                          | Tree hierarchies of objects & groups  |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Enterprise Banking Operations Facade (`BankingSystemFacade`)
> **Core Principle**:
> - A complex banking transfer involves multiple distinct subsystems: `AccountRepository`, `LedgerAuditService`, `NotificationGateway`, and `FraudDetectionEngine`.
> - A `BankingSystemFacade` hides all these subsystems behind a clean, single-call interface.

In [ ]:
# Banking Transfer Facade
class AccountService:
    def __init__(self):
        self.balances = {"ACC-101": 50000.0, "ACC-202": 12000.0}
    
    def debit(self, acc, amt):
        if self.balances[acc] < amt:
            raise ValueError("Insufficient funds")
        self.balances[acc] -= amt
        
    def credit(self, acc, amt):
        self.balances[acc] += amt

class AuditLogger:
    def record(self, sender, receiver, amt):
        print(f"  [AUDIT] Logged transaction: {sender} -> {receiver} (Rs. {amt:,.2f})")

class SMSNotificationService:
    def notify(self, acc, msg):
        print(f"  [SMS SENT to {acc}]: {msg}")

class BankingSystemFacade:
    def __init__(self):
        self.accounts = AccountService()
        self.audit = AuditLogger()
        self.sms = SMSNotificationService()

    def transfer_funds(self, source, target, amount):
        print(f"Initiating Fund Transfer: Rs. {amount:,.2f} from {source} to {target}")
        self.accounts.debit(source, amount)
        self.accounts.credit(target, amount)
        self.audit.record(source, target, amount)
        self.sms.notify(source, f"Debited Rs. {amount:,.2f}. Balance: Rs. {self.accounts.balances[source]:,.2f}")
        self.sms.notify(target, f"Credited Rs. {amount:,.2f}. Balance: Rs. {self.accounts.balances[target]:,.2f}")
        print("Transfer executed seamlessly via Facade!\n")

bank_facade = BankingSystemFacade()
bank_facade.transfer_funds("ACC-101", "ACC-202", 8500.0)

### 2. Legacy Data Format Adapter
Adapting older comma-delimited raw strings into modern dictionary structures expected by reporting engines.

In [ ]:
# Legacy String Data Adapter
class LegacyDataFeed:
    def get_raw_student_records(self):
        return [
            "101,Anjali,Python Full Stack,94.0",
            "102,Brijesh,Data Science,86.5",
            "103,Chitra,Cloud DevOps,78.0"
        ]

class StudentDataFeedAdapter:
    def __init__(self, legacy_feed):
        self.legacy_feed = legacy_feed

    def get_structured_records(self) -> list[dict]:
        structured = []
        for line in self.legacy_feed.get_raw_student_records():
            sno, name, course, marks = line.split(",")
            structured.append({
                "student_id": int(sno),
                "name": name,
                "course": course,
                "marks": float(marks)
            })
        return structured

adapter = StudentDataFeedAdapter(LegacyDataFeed())
for record in adapter.get_structured_records():
    print("Parsed via Adapter:", record)